# 73 · 정정된 검색 지표와 수동 답변 평가 검증

73번 최초 노트북은 유료 실행 전 입력·순위 결과를 고정한 원본입니다. 이후 C01의 임직원 전용 카드에 대한 자격 미확인을 발견해 정정했으므로, 최종 검색 지표는 `ranking_summary_corrected.json`을 사용합니다. 최초 노트북과 정정 전 자료는 이력으로 보존합니다. 새 API 호출 없이 실제 자료와 수동 판정을 다시 결속합니다.

In [1]:
from pathlib import Path
import collections, hashlib, importlib.util, json, math, statistics
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
BASE=ROOT/'notebooks/data/73_repaired_card_scope'; OLD=ROOT/'notebooks/data/70_cap_context_repair'
def read(path): return json.loads(Path(path).read_text())
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def load(name,path):
    s=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(s);s.loader.exec_module(m);return m
rank=load('rank73_final',ROOT/'notebooks/scripts/73_card_ranking.py')
eval73=load('eval73_final',ROOT/'notebooks/scripts/73_answer_evaluation.py')
rt=load('runtime73_final',ROOT/'notebooks/scripts/73_answer_runtime.py')
print('최종 평가: 질문 12개, 프로필 2개, 반복 2회; 새 API 호출 없음')

최종 평가: 질문 12개, 프로필 2개, 반복 2회; 새 API 호출 없음


## 1. 원문 발급 자격 정정과 표준 검색 지표 재계산

KB Star B는 KB금융그룹 임직원 전용입니다. 사용자의 발급 자격이 제시되지 않아 C01의 해당 카드를 적합으로 확정하지 않습니다. 원래 73 준비 파일을 덮어쓰지 않고 정정 파일로 다시 계산합니다.

In [2]:
correction=read(BASE/'source_label_correction.json')['records'][0]
assert correction['query_id']=='C01' and correction['previous'] is True and correction['corrected'] is None
assert sha(ROOT/correction['pdf_path'])==correction['pdf_sha256']
import fitz
with fitz.open(ROOT/correction['pdf_path']) as pdf:
    assert correction['quote'] in pdf[correction['page']-1].get_text()
labels=read(BASE/'query_gold_corrected.json')['labels'];qids=set(labels)
assert qids==set(rt.QUERIES) and labels['C01'][correction['card_key']] is None
unknown_queries=sorted(q for q,pool in labels.items() if None in pool.values())
assert unknown_queries==['C01','C11','C14']
prior=read(OLD/'retrieval_results.json')['records'];result=[]
for r in prior:
    q=r['query_id']
    if q not in qids or r['repeat'] not in (1,2): continue
    cards=[c['card_key'] for c in r['result']['cards']]
    result.append(dict(identity=r['identity'],query_id=q,profile=r['profile'],mode=r['mode'],repeat=r['repeat'],
                       cards=cards,**rank.score(cards,labels[q],3)))
assert len(result)==144 and result==read(BASE/'ranking_results_corrected.json')['records']
summary=read(BASE/'ranking_summary_corrected.json');metrics=('hit_lower','hit_upper','mrr_lower','mrr_upper','recall_judged_pool','ndcg_judged_pool')
for row in summary['records']:
    group=[r for r in result if r['profile']==row['profile'] and r['mode']==row['mode']]
    for metric in metrics:
        qmeans=[]
        for q in sorted(qids):
            values=[r[metric] for r in group if r['query_id']==q]
            assert len(values)==2
            if all(x is not None for x in values):qmeans.append(statistics.mean(values))
        assert row[metric+'_questions']==len(qmeans)
        assert math.isclose(row[metric],statistics.mean(qmeans),abs_tol=1e-12)
print('카드 순위 144행 재계산 일치. Hit/MRR 12질문, 공통 후보 Recall/nDCG 9질문.')
for s in summary['records']:
    print(s['profile'],s['mode'],'Hit',round(s['hit_lower'],4),'~',round(s['hit_upper'],4),
          'MRR',round(s['mrr_lower'],4),'~',round(s['mrr_upper'],4),
          '후보집합 Recall',round(s['recall_judged_pool'],4),'nDCG',round(s['ndcg_judged_pool'],4))

카드 순위 144행 재계산 일치. Hit/MRR 12질문, 공통 후보 Recall/nDCG 9질문.
card_page_section_benefit U Hit 0.9167 ~ 1 MRR 0.9167 ~ 0.9583 후보집합 Recall 0.6923 nDCG 0.9081
card_page_section_benefit SEQ65 Hit 1 ~ 1 MRR 1.0 ~ 1.0 후보집합 Recall 0.7201 nDCG 0.9479
card_page_section_benefit SEQ_REPAIRED Hit 1 ~ 1 MRR 1.0 ~ 1.0 후보집합 Recall 0.7201 nDCG 0.9479
parent_child_bundle U Hit 1 ~ 1 MRR 1.0 ~ 1.0 후보집합 Recall 0.6701 nDCG 0.8752
parent_child_bundle SEQ65 Hit 1 ~ 1 MRR 1.0 ~ 1.0 후보집합 Recall 0.7201 nDCG 0.9479
parent_child_bundle SEQ_REPAIRED Hit 1 ~ 1 MRR 1.0 ~ 1.0 후보집합 Recall 0.7201 nDCG 0.9479


## 2. 표시된 답변의 수동 판단 결속

73 신규 답변은 실제 본문·추천 이유를 원문과 입력에 대조한 수동 판단입니다. 기존 70의 질문별 판정을 재사용하되 C01 임직원 발급 자격 누락 두 건에만 동일 기준의 보조 재판정을 적용합니다. 기존70 점수는 변경하지 않습니다.

In [3]:
jobs={j['identity']:j for j in read(BASE/'jobs.json')['records']}
answers={a['identity']:a for a in read(BASE/'execution_results.json')['records']}
notes=read(BASE/'manual_review_notes.json')['records'];combined=read(BASE/'answer_evaluation_results.json')
rt.validate_matrix(list(jobs.values()));assert len(answers)==len(notes)==48 and set(jobs)==set(answers)==set(notes)
new=[r for r in combined['records'] if not r['historic']];historic=[r for r in combined['records'] if r['historic']]
assert len(new)==48 and len(historic)==96
for saved in new:
    got=eval73.score(jobs[saved['identity']],answers[saved['identity']],notes[saved['identity']])
    assert saved=={**got,'historic':False}
old_scores={r['identity']:r for r in read(OLD/'evaluation_results.json')['records']}
old_notes={r['identity']:r for r in read(OLD/'manual_reviews.json')['records']}
changes={r['identity']:r for r in read(BASE/'baseline_eligibility_sensitivity.json')['records']}
assert len(changes)==2
for saved in historic:
    old=old_scores[saved['identity']]
    assert saved['technical']==old['technical_success'] and saved['direct']==old['direct_request_satisfied']
    if saved['identity'] in changes:
        assert old['passed'] and not saved['passed'] and saved['uncertain']
        row=read(OLD/'answers'/(saved['identity']+'.json'))
        assert eval73.digest(row)==changes[saved['identity']]['previous_row_sha256']
    else:
        assert saved['passed']==old['passed'] and saved['accurate']==old['visible_factual_accuracy']
        assert saved['uncertain']==(old_notes[saved['identity']].get('provisional_visible_accuracy')=='uncertain' and saved['technical'])
for s in combined['summary']:
    group=[r for r in combined['records'] if r['profile']==s['profile'] and r['mode']==s['mode']]
    assert len(group)==24
    for key in ('technical','direct','accurate','grounded','passed','confirmed_wrong','uncertain'):
        assert s[key]==sum(bool(r[key]) for r in group)
print('신규 48개 수동 판단과 기존 96개 답변 재결속 완료. 의미 판단은 수동·비맹검.')
for s in combined['summary']:
    print(s['profile'],s['mode'],'최종',s['passed'],'/24, 직접',s['direct'],'원문정확',s['accurate'],
          '확정오류',s['confirmed_wrong'],'불확실',s['uncertain'])

신규 48개 수동 판단과 기존 96개 답변 재결속 완료. 의미 판단은 수동·비맹검.
card_page_section_benefit U 최종 22 /24, 직접 23 원문정확 22 확정오류 0 불확실 1
card_page_section_benefit SEQ65 최종 22 /24, 직접 24 원문정확 22 확정오류 0 불확실 2
card_page_section_benefit SEQ_CARDS 최종 17 /24, 직접 23 원문정확 17 확정오류 2 불확실 5
parent_child_bundle U 최종 21 /24, 직접 24 원문정확 21 확정오류 0 불확실 3
parent_child_bundle SEQ65 최종 22 /24, 직접 23 원문정확 22 확정오류 1 불확실 1
parent_child_bundle SEQ_CARDS 최종 20 /24, 직접 23 원문정확 21 확정오류 1 불확실 2


## 3. 호출·비용·원본 보존

49회 물리 요청의 사용량을 원본 응답에서 재합산합니다. 사전 게이트와 기존 원본 보존 파일을 검사하되 유료 실행기 `run()`은 호출하지 않습니다.

In [4]:
rt.require_gate();manifest=read(BASE/'execution_manifest.json');rt.T.guard_manifest(manifest)
joblist=list(jobs.values());assert rt.A.payload_contract(joblist,manifest)==read(BASE/'answer_payload_contract.json')
for job in joblist:rt.A.validate_answer(answers[job['identity']],job,manifest,read(BASE/'answer_payload_contract.json'),BASE)
protected=rt.T.E.verify_file_map(read(BASE/'preservation_before.json')['files_sha256'],ROOT,fresh=True)
files=list((BASE/'api_cache').glob('*.json'));keys=[e['key'] for a in answers.values() for e in a['events']]
assert len(files)==len(keys)==49 and len(set(keys))==49 and set(keys)=={p.stem for p in files}
priced=rt.T.R.P.PRICE.price_records([('73',str(p.relative_to(ROOT)),read(p)) for p in files]);totals=rt.T.R.P.PRICE.aggregate(priced)
saved=read(BASE/'cost_latency_analysis.json');assert totals==saved['new_stage'] and totals['unknown_cost_records']==0
ledger=rt.T.budget_ledger();assert ledger['budget_debit_usd']==saved['cumulative_budget_debit_usd']
print('보호 원본:',protected,'파일; 실제 호출:',len(files),'회; 신규 환산 USD:',totals['known_cost_usd'], '누적 차감 USD:',ledger['budget_debit_usd'])
print('실제 청구액·첫 토큰·화면 표시 시간·운영 전체 지연은 미측정.')

보호 원본: 3431 파일; 실제 호출: 49 회; 신규 환산 USD: 0.054072115 누적 차감 USD: 2.826878200
실제 청구액·첫 토큰·화면 표시 시간·운영 전체 지연은 미측정.
